# Improving the precision and recall of the curation model: pooled regimes and PCA-based metrics

Two ideas, tested with nested cross-validation (test units never used to choose anything):

1. **pooled training**: train on spontaneous (NS) AND evoked (E) units together, test on each regime separately, against a model trained on that regime alone;
2. **PCA-based quality metrics** that the paper left out as expensive (isolation distance, L ratio, d prime; the nearest-neighbour metrics are too slow, see the setup cell), alone and with the template/waveform features.

**validation**: outer loop = repeated stratified 4-fold (75% / 25%); inner loop (training units only) = 5-fold CV that selects the hyper-parameters AND the decision threshold (maximizing F1). Every model sees the same outer folds, so differences are paired. Baseline = current model (unpenalized GLM on the 10 quality metrics, threshold 0.8) trained on the same regime. `avg_precision` is the threshold-free area under the precision-recall curve.

Requires the waveforms saved by fig5e.ipynb and fig5f.ipynb.

In [1]:
%load_ext autoreload
%autoreload 2
import os, shutil, warnings
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
import numpy as np
import pandas as pd
import spikeinterface as si
import spikecurate
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn import metrics as skm
from spikeinterface.core import get_template_extremum_channel
from spikeinterface.qualitymetrics import compute_quality_metrics
from spikeinterface.qualitymetrics.misc_metrics import compute_amplitude_medians
warnings.filterwarnings("ignore")

os.chdir("/home/steeve/steeve/epfl/code/spikebias/")
import sys; sys.path.insert(0, os.getcwd())
from src.nodes.models.Flc import dataloader

QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"
WE_JOB_KWARGS = dict(n_jobs=8, chunk_duration="1s", progress_bar=False)
LABEL = "quality_label"
REGIMES = {  # name: (experiment code in sorting_quality.csv, waveform folder, 10 min ground truth)
    "NS": ("NS", "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m_single_units", "dataset/00_raw/ground_truth_npx_spont_10m"),
    "E":  ("E",  "dataset/01_intermediate/waveforms/npx_evoked/SortingKS4_10m_single_units", "dataset/00_raw/ground_truth_npx_evoked_10m"),
}
# nearest_neighbor, nn_isolation and nn_noise_overlap are left out: nearest_neighbor alone took ~9 min of 16 cores for 184 units (and scales with the square of the number of units)
PC_METRICS = ["isolation_distance", "l_ratio", "d_prime"]
VIEW_DIR = "temp/improve_model_views"       # symlink views, so that nothing is written in the waveform folders

## Datasets (10 quality metrics + template/waveform features + PCA-based metrics) for both regimes

In [2]:
def template_features(We):
    fs = We.sampling_frequency
    ext_ch = get_template_extremum_channel(We, peak_sign="neg", outputs="index")
    amp_med = compute_amplitude_medians(We)
    rows = {}
    for u in We.unit_ids:
        ch = ext_ch[u]
        inds = We.sparsity.unit_id_to_channel_indices[u]
        pos = int(np.where(inds == ch)[0][0])
        tmpl = We.get_template(u, mode="average")
        tmpl = tmpl[:, pos] if tmpl.shape[1] == len(inds) else tmpl[:, ch]
        trough = int(np.argmin(tmpl)); peak = trough + int(np.argmax(tmpl[trough:]))
        w = We.get_waveforms(u)[:, :, pos]; ta = w[:, trough]
        corr = np.array([np.corrcoef(wi, tmpl)[0, 1] for wi in w])
        rows[int(u)] = dict(amp_template=-tmpl[trough], width_ms=(peak - trough) / fs * 1000, n_channels=len(inds),
                            trough_cv=np.std(ta) / abs(np.mean(ta)), shape_corr_med=np.nanmedian(corr),
                            shape_lowcorr=np.nanmean(corr < 0.7), amp_median=amp_med[u])
    return pd.DataFrame.from_dict(rows, orient="index")

def pc_metrics(waveform_path, name):
    """PCA-based quality metrics, computed on a symlink view of the waveform folder (nothing is written there)"""
    view = f"{VIEW_DIR}/{name}"
    shutil.rmtree(view, ignore_errors=True); os.makedirs(view)
    for f in os.listdir(waveform_path):
        if f != "quality_metrics":
            os.symlink(os.path.abspath(f"{waveform_path}/{f}"), f"{view}/{f}")
    we = si.load_waveforms(view, with_recording=False)
    r = compute_quality_metrics(we, metric_names=PC_METRICS, skip_pc_metrics=False, n_jobs=16, progress_bar=False)
    r.index = r.index.astype(int)
    return r

parts, EXTRA = [], None
for name, (exp, wpath, gt) in REGIMES.items():
    We = si.load_waveforms(wpath, with_recording=True)
    ids = np.array([int(u) for u in We.unit_ids])
    good, bad = dataloader.get_good_and_bad_units(QUALITY_PATH, exp=exp, sorter="KS4")
    data = spikecurate.load_dataset(ids, good, bad, We.sorting, si.load_extractor(gt), We,
                                    load_qm_if_exists=False, delta_time=1.3, job_kwargs=WE_JOB_KWARGS)
    base = list(data["predictors"])
    d = data["dataset"].join(template_features(We)).join(pc_metrics(wpath, name))
    d["regime"] = name
    parts.append(d); print(name, d.shape, "good:", int(d[LABEL].sum()))
dataset = pd.concat(parts)
dataset = dataset.dropna(subset=base + ["amp_template"])          # units with a missing base metric (silhouette)
EXTRA = ["amp_template", "width_ms", "n_channels", "trough_cv", "shape_corr_med", "shape_lowcorr", "amp_median"]
PCM = ["isolation_distance", "l_ratio", "d_prime"]
PCM = [c for c in PCM if c in dataset.columns]
print(dataset.groupby("regime")[LABEL].agg(["count", "sum"]))
print("missing PCA-metric values:", dataset[PCM].isna().sum().to_dict())

NS (184, 22) good: 84


E (408, 22) good: 105
        count    sum
regime              
E         407  105.0
NS        182   84.0
missing PCA-metric values: {'isolation_distance': 1, 'l_ratio': 16, 'd_prime': 0}


## Nested cross-validation machinery

In [3]:
INNER_SPLITS = 5
def pipe(est, scale=False):
    steps = [SimpleImputer(strategy="median")] + ([StandardScaler()] if scale else []) + [est]
    return make_pipeline(*steps)

def scores(m, X):
    return m.predict_proba(X)[:, 1] if hasattr(m, "predict_proba") else m.decision_function(X)

def best_threshold(y_true, s):
    cand = np.unique(s); cand = (cand[:-1] + cand[1:]) / 2 if len(cand) > 1 else cand
    f1 = [skm.f1_score(y_true, (s >= t).astype(int), zero_division=0) for t in cand]
    k = int(np.argmax(f1)); return cand[k], f1[k]

def tuned(estimator, grid, Xtr, ytr, seed):
    skf = StratifiedKFold(INNER_SPLITS, shuffle=True, random_state=seed)
    best = None
    for params in ParameterGrid(grid):
        oof = np.zeros(len(ytr))
        for a, b in skf.split(Xtr, ytr):
            m = clone(estimator).set_params(**params).fit(Xtr[a], ytr[a]); oof[b] = scores(m, Xtr[b])
        t, f1 = best_threshold(ytr, oof)
        if best is None or f1 > best[0]: best = (f1, params, t)
    _, params, t = best
    return clone(estimator).set_params(**params).fit(Xtr, ytr), t

MODELS = {
    "logit + tuned thr": (pipe(LogisticRegression(penalty=None, max_iter=5000), True), {}),
    "L1 logit":          (pipe(LogisticRegression(penalty="l1", solver="liblinear", max_iter=5000), True), {"logisticregression__C": [0.1, 0.3, 1, 3, 10]}),
    "gradient boosting": (pipe(GradientBoostingClassifier(random_state=0)), {"gradientboostingclassifier__n_estimators": [100, 300], "gradientboostingclassifier__max_depth": [1, 2, 3], "gradientboostingclassifier__learning_rate": [0.05]}),
    "random forest":     (pipe(RandomForestClassifier(n_estimators=300, random_state=0)), {"randomforestclassifier__min_samples_leaf": [1, 3, 5]}),
    "extra trees":       (pipe(ExtraTreesClassifier(n_estimators=300, random_state=0)), {"extratreesclassifier__min_samples_leaf": [1, 3, 5]}),
    "SVM (RBF)":         (pipe(SVC(), True), {"svc__C": [0.3, 1, 3, 10], "svc__gamma": ["scale", 0.03]}),
}
BASELINE = "baseline (GLM, thr 0.8, own regime)"

def prf(y, s, thr):
    p = (s >= thr).astype(int)
    return (skm.precision_score(y, p, zero_division=0), skm.recall_score(y, p, zero_division=0), skm.average_precision_score(y, s))

def one_fold(k, tr, te, df, cols, train_regimes, base_cols):
    """train on units of `train_regimes` (within the training fold); test on each regime separately"""
    X = df[cols].values; Xb = df[base_cols].values; y = df[LABEL].values.astype(int); rg = df["regime"].values
    trm = tr[np.isin(rg[tr], train_regimes)]
    out = {}
    for r in np.unique(rg[te]):
        tr_r = tr[rg[tr] == r]; te_r = te[rg[te] == r]
        b, _ = tuned(pipe(LogisticRegression(penalty=None, max_iter=5000), True), {}, Xb[tr_r], y[tr_r], k)
        out[(BASELINE, r)] = prf(y[te_r], scores(b, Xb[te_r]), 0.8)
    for name, (est, grid) in MODELS.items():
        m, t = tuned(est, grid, X[trm], y[trm], k)
        for r in np.unique(rg[te]):
            te_r = te[rg[te] == r]
            out[(name, r)] = prf(y[te_r], scores(m, X[te_r]), t)
    return out

def run(df, cols, train_regimes, base_cols, n_repeats=10):
    strat = (df["regime"].astype(str) + df[LABEL].astype(int).astype(str)).values
    folds = list(RepeatedStratifiedKFold(n_splits=4, n_repeats=n_repeats, random_state=0).split(np.zeros(len(df)), strat))
    res = Parallel(n_jobs=16)(delayed(one_fold)(k, tr, te, df, cols, train_regimes, base_cols) for k, (tr, te) in enumerate(folds))
    return {key: np.array([r[key] for r in res]) for key in res[0]}

def summarize(res, regime):
    base = res[(BASELINE, regime)]; ci = lambda x: 1.96 * np.std(x) / np.sqrt(len(x)); rows = []
    for (n, r), a in res.items():
        if r != regime: continue
        p, rc = a[:, 0], a[:, 1]; d = a - base
        rows.append({"model": n, "precision": np.median(p), "recall": np.median(rc), "F1": np.median(2 * p * rc / np.maximum(p + rc, 1e-9)),
                     "avg_precision": np.median(a[:, 2]), "dPrec": d[:, 0].mean(), "dP_ci": ci(d[:, 0]), "dRec": d[:, 1].mean(), "dR_ci": ci(d[:, 1]),
                     "dAP": d[:, 2].mean(), "dAP_ci": ci(d[:, 2])})
    t = pd.DataFrame(rows).set_index("model")
    t["both up"] = (t["dPrec"] - t["dP_ci"] > 0) & (t["dRec"] - t["dR_ci"] > 0)
    t["AP up"] = t["dAP"] - t["dAP_ci"] > 0
    return t.round(3)

## Idea 2: PCA-based metrics (spontaneous regime only)

In [4]:
%%time
ns = dataset[dataset["regime"] == "NS"].reset_index(drop=True)
sets = {"A: 10 quality metrics": base, "C: 10 + PCA-based metrics": base + PCM, "D: 10 + PCA-based + template/waveform": base + PCM + EXTRA}
summ2 = {}
for label, cols in sets.items():
    res = run(ns, cols, ["NS"], base)
    summ2[label] = summarize(res, "NS"); print("\n###", label); display(summ2[label])


### A: 10 quality metrics


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.879,0.619,0.706,0.895,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.745,0.905,0.792,0.895,-0.147,0.022,0.261,0.032,0.000,0.000,False,False
L1 logit,0.731,0.905,0.816,0.903,-0.154,0.022,0.312,0.032,0.007,0.008,False,False
gradient boosting,0.792,0.905,0.833,0.902,-0.095,0.031,0.265,0.029,0.002,0.018,False,False
random forest,0.825,0.881,0.833,0.916,-0.072,0.029,0.263,0.035,0.014,0.018,False,False
extra trees,0.805,0.905,0.848,0.932,-0.059,0.025,0.280,0.031,0.028,0.014,False,True
SVM (RBF),0.800,0.905,0.830,0.911,-0.092,0.024,0.267,0.027,0.016,0.012,False,True



### C: 10 + PCA-based metrics


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.879,0.619,0.706,0.895,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.750,0.857,0.788,0.894,-0.148,0.030,0.261,0.033,-0.016,0.014,False,False
L1 logit,0.714,0.905,0.804,0.875,-0.160,0.025,0.310,0.028,-0.022,0.019,False,False
gradient boosting,0.800,0.905,0.831,0.918,-0.090,0.033,0.281,0.032,0.011,0.019,False,False
random forest,0.769,0.905,0.835,0.913,-0.109,0.029,0.299,0.026,0.008,0.019,False,False
extra trees,0.826,0.905,0.864,0.921,-0.066,0.025,0.306,0.030,0.019,0.018,False,True
SVM (RBF),0.778,0.857,0.800,0.904,-0.107,0.027,0.248,0.029,-0.001,0.016,False,False



### D: 10 + PCA-based + template/waveform


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.879,0.619,0.706,0.895,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.721,0.881,0.792,0.875,-0.162,0.023,0.279,0.038,-0.026,0.017,False,False
L1 logit,0.760,0.905,0.822,0.879,-0.140,0.022,0.301,0.030,-0.028,0.018,False,False
gradient boosting,0.750,0.905,0.817,0.898,-0.132,0.027,0.302,0.028,0.000,0.018,False,False
random forest,0.765,0.905,0.828,0.907,-0.110,0.029,0.305,0.032,0.012,0.017,False,False
extra trees,0.800,0.905,0.848,0.903,-0.089,0.027,0.307,0.030,0.005,0.018,False,False
SVM (RBF),0.771,0.857,0.813,0.898,-0.110,0.028,0.267,0.027,0.001,0.017,False,False


CPU times: user 490 ms, sys: 644 ms, total: 1.13 s
Wall time: 1min 41s


## Idea 1: pooled training (spontaneous + evoked), tested on each regime

In [5]:
%%time
dataset["is_evoked"] = (dataset["regime"] == "E").astype(int)
df = dataset.reset_index(drop=True)
for label, cols, train_regimes in [
        ("own-regime training, 10 metrics (reference)", base, None),
        ("pooled, 10 metrics", base, ["NS", "E"]),
        ("pooled, 10 metrics + regime indicator", base + ["is_evoked"], ["NS", "E"]),
        ("pooled, 10 + PCA-based metrics", base + PCM + ["is_evoked"], ["NS", "E"])]:
    if train_regimes is None:
        # reference: models trained on the test regime only -> run once per regime
        res = {}
        for r in ["NS", "E"]:
            sub = df[df["regime"] == r].reset_index(drop=True)
            rr = run(sub, cols, [r], base, n_repeats=5)
            res.update(rr)
    else:
        res = run(df, cols, train_regimes, base, n_repeats=5)
    for r in ["NS", "E"]:
        print(f"\n### {label} | tested on {r}"); display(summarize(res, r))


### own-regime training, 10 metrics (reference) | tested on NS


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.902,0.595,0.706,0.897,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.755,0.905,0.800,0.897,-0.152,0.032,0.267,0.041,0.000,0.000,False,False
L1 logit,0.750,0.929,0.818,0.907,-0.167,0.023,0.324,0.037,0.005,0.013,False,False
gradient boosting,0.809,0.857,0.835,0.915,-0.099,0.045,0.252,0.034,-0.003,0.024,False,False
random forest,0.845,0.857,0.842,0.928,-0.080,0.033,0.260,0.044,0.014,0.023,False,False
extra trees,0.842,0.905,0.854,0.936,-0.064,0.037,0.267,0.039,0.025,0.022,False,True
SVM (RBF),0.800,0.905,0.834,0.917,-0.095,0.032,0.274,0.032,0.016,0.019,False,False



### own-regime training, 10 metrics (reference) | tested on E


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.938,0.547,0.659,0.865,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.706,0.865,0.778,0.865,-0.201,0.036,0.332,0.056,0.000,0.000,False,False
L1 logit,0.712,0.868,0.786,0.850,-0.209,0.031,0.350,0.049,-0.018,0.016,False,False
gradient boosting,0.734,0.868,0.800,0.898,-0.175,0.038,0.353,0.052,0.019,0.020,False,False
random forest,0.764,0.846,0.800,0.914,-0.153,0.039,0.307,0.045,0.031,0.016,False,True
extra trees,0.757,0.906,0.803,0.904,-0.157,0.044,0.370,0.059,0.032,0.015,False,True
SVM (RBF),0.742,0.923,0.800,0.870,-0.186,0.040,0.362,0.051,0.005,0.015,False,False



### pooled, 10 metrics | tested on NS


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.897,0.619,0.712,0.887,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.760,0.833,0.787,0.856,-0.144,0.030,0.245,0.050,-0.046,0.020,False,False
L1 logit,0.735,0.905,0.809,0.825,-0.158,0.032,0.305,0.056,-0.064,0.023,False,False
gradient boosting,0.818,0.881,0.839,0.908,-0.088,0.041,0.293,0.055,0.027,0.013,False,True
random forest,0.804,0.857,0.848,0.924,-0.092,0.031,0.279,0.071,0.032,0.016,False,True
extra trees,0.809,0.905,0.841,0.916,-0.099,0.040,0.319,0.065,0.031,0.016,False,True
SVM (RBF),0.774,0.905,0.837,0.906,-0.122,0.041,0.326,0.052,0.013,0.017,False,False


... (7 outputs left out)

,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.897,0.619,0.712,0.887,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.745,0.857,0.800,0.837,-0.150,0.033,0.290,0.049,-0.050,0.020,False,False
L1 logit,0.727,0.952,0.821,0.824,-0.172,0.029,0.362,0.048,-0.054,0.018,False,False
gradient boosting,0.795,0.881,0.810,0.895,-0.123,0.033,0.281,0.059,0.005,0.016,False,False
random forest,0.769,0.952,0.832,0.898,-0.129,0.027,0.343,0.064,0.011,0.015,False,False
extra trees,0.813,0.952,0.840,0.904,-0.104,0.033,0.345,0.056,0.021,0.013,False,True
SVM (RBF),0.771,0.857,0.818,0.880,-0.143,0.038,0.305,0.057,0.002,0.015,False,False



### pooled, 10 + PCA-based metrics | tested on E


,precision,recall,F1,avg_precision,dPrec,dP_ci,dRec,dR_ci,dAP,dAP_ci,both up,AP up
model,,,,,,,,,,,,
"baseline (GLM, thr 0.8, own regime)",0.882,0.500,0.643,0.855,0.000,0.000,0.000,0.000,0.000,0.000,False,False
logit + tuned thr,0.772,0.885,0.823,0.889,-0.103,0.042,0.335,0.034,0.032,0.017,False,True
L1 logit,0.750,0.923,0.825,0.897,-0.137,0.049,0.393,0.035,0.050,0.018,False,True
gradient boosting,0.763,0.923,0.836,0.923,-0.109,0.055,0.391,0.039,0.071,0.020,False,True
random forest,0.766,0.942,0.839,0.926,-0.115,0.048,0.408,0.039,0.074,0.023,False,True
extra trees,0.783,0.906,0.829,0.933,-0.087,0.046,0.379,0.038,0.078,0.022,False,True
SVM (RBF),0.782,0.852,0.811,0.907,-0.109,0.039,0.339,0.032,0.053,0.018,False,True


CPU times: user 495 ms, sys: 155 ms, total: 649 ms
Wall time: 2min 21s
